In [2]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CATEGORIES
from src.utils.io import read_jsonl_sample

CATEGORY = 'history_biography'
cfg = CATEGORIES[CATEGORY]
OUTPUT_DIR = cfg.processed_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Category: {cfg.display_name}')
print(f'Books file: {cfg.books_file}')
print(f'Interactions file: {cfg.interactions_file}')
print(f'Output dir: {OUTPUT_DIR}')


Category: History & Biography
Books file: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\raw\goodreads_books_history_biography.json.gz
Interactions file: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\raw\goodreads_interactions_history_biography.json.gz
Output dir: C:\Users\PC\OneDrive\Desktop\book-recommendations\data\interim\history_biography


---
# Books

In [3]:
SAMPLE_SIZE = 5_000
books_sample = read_jsonl_sample(cfg.books_file, nrows=SAMPLE_SIZE)
print(f'Shape: {books_sample.shape}')
print(f'Columns ({len(books_sample.columns)}):')
print(list(books_sample.columns))

Shape: (5000, 29)
Columns (29):
['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code', 'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin', 'similar_books', 'description', 'format', 'link', 'authors', 'publisher', 'num_pages', 'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'url', 'image_url', 'book_id', 'ratings_count', 'work_id', 'title', 'title_without_series']


In [4]:
summary_books = pd.DataFrame({
    'dtype': books_sample.dtypes.astype(str),
    'non_null': books_sample.notna().sum(),
    'null_count': books_sample.isna().sum(),
    'null_pct': (books_sample.isna().sum() / len(books_sample) * 100).round(1),
    'n_unique': books_sample.apply(lambda col: col.map(str).nunique()),
    'example': books_sample.iloc[0].astype(str).str[:80],
})
display(summary_books)

,dtype,non_null,null_count,null_pct,n_unique,example
isbn,str,5000,0,0.0,3620,1599150603
text_reviews_count,str,5000,0,0.0,296,7
series,object,5000,0,0.0,1169,[]
country_code,str,5000,0,0.0,1,US
language_code,str,5000,0,0.0,53,
popular_shelves,object,5000,0,0.0,4774,"[{'count': '56', 'name': 'to-read'}, {'count':..."
asin,str,5000,0,0.0,677,
is_ebook,str,5000,0,0.0,2,false
average_rating,str,5000,0,0.0,210,4.13
kindle_asin,str,5000,0,0.0,2349,B00DU10PUG


In [5]:
books_sample.head(3)

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,1599150603,7,[],US,,"[{'count': '56', 'name': 'to-read'}, {'count':...",,false,4.13,B00DU10PUG,...,9,,2006,https://www.goodreads.com/book/show/287141.The...,https://s.gr-assets.com/assets/nophoto/book/11...,287141,46,278578,The Aeneid for Boys and Girls,The Aeneid for Boys and Girls
1,184737297X,15,[169353],US,,"[{'count': '159', 'name': 'to-read'}, {'count'...",,false,3.93,B007YLTG5I,...,4,,2009,https://www.goodreads.com/book/show/6066814-cr...,https://images.gr-assets.com/books/1328724803m...,6066814,186,6243149,"Crowner Royal (Crowner John Mystery, #13)","Crowner Royal (Crowner John Mystery, #13)"
2,037583687X,615,[],US,,"[{'count': '4248', 'name': 'to-read'}, {'count...",,false,3.98,B0010SEMV4,...,7,,2006,https://www.goodreads.com/book/show/89377.Penn...,https://images.gr-assets.com/books/1320470906m...,89377,6949,86258,Penny from Heaven,Penny from Heaven


In [6]:
BOOKS_COLUMNS_TO_DROP = [
    'isbn',
    'isbn13',
    'asin',
    'kindle_asin',
    'url',
    'link',
    'image_url',
    'country_code',
    'edition_information',
    'publication_day',
    'publication_month',
    'similar_books',
    'title_without_series',
    'is_ebook',
    'description',
    'publisher',
    'format'
]

In [7]:
books_clean = books_sample.drop(columns=[c for c in BOOKS_COLUMNS_TO_DROP if c in books_sample.columns])

In [8]:
summary_clean = pd.DataFrame({
    'dtype': books_clean.dtypes.astype(str),
    'non_null': books_clean.notna().sum(),
    'null_pct': (books_clean.isna().sum() / len(books_clean) * 100).round(1),
})
display(summary_clean)

,dtype,non_null,null_pct
text_reviews_count,str,5000,0.0
series,object,5000,0.0
language_code,str,5000,0.0
popular_shelves,object,5000,0.0
average_rating,str,5000,0.0
authors,object,5000,0.0
num_pages,str,5000,0.0
publication_year,str,5000,0.0
book_id,str,5000,0.0
ratings_count,str,5000,0.0


In [9]:
books_clean.head(5)

,text_reviews_count,series,language_code,popular_shelves,average_rating,authors,num_pages,publication_year,book_id,ratings_count,work_id,title
0,7,[],,"[{'count': '56', 'name': 'to-read'}, {'count':...",4.13,"[{'author_id': '3041852', 'role': ''}]",162,2006,287141,46,278578,The Aeneid for Boys and Girls
1,15,[169353],,"[{'count': '159', 'name': 'to-read'}, {'count'...",3.93,"[{'author_id': '37778', 'role': ''}]",400,2009,6066814,186,6243149,"Crowner Royal (Crowner John Mystery, #13)"
2,615,[],,"[{'count': '4248', 'name': 'to-read'}, {'count...",3.98,"[{'author_id': '137561', 'role': ''}]",288,2006,89377,6949,86258,Penny from Heaven
3,44,[],en-US,"[{'count': '362', 'name': 'to-read'}, {'count'...",3.75,"[{'author_id': '31308', 'role': ''}]",,,6158967,338,6338156,Crude World: The Violent Twilight of Oil
4,19,[],ita,"[{'count': '6463', 'name': 'to-read'}, {'count...",4.28,"[{'author_id': '51229', 'role': ''}, {'author_...",332,2012,18628480,116,1559207,Stoner


In [10]:
from src.utils.io import read_jsonl_chunks
import pyarrow as pa
import pyarrow.parquet as pq

BOOKS_OUTPUT_PATH = OUTPUT_DIR / 'books_clean.parquet'
CHUNKSIZE = 50_000

writer = None
total_rows = 0

for chunk_idx, chunk in enumerate(read_jsonl_chunks(cfg.books_file, chunksize=CHUNKSIZE)):
    chunk = chunk.drop(columns=[c for c in BOOKS_COLUMNS_TO_DROP if c in chunk.columns])
    total_rows += len(chunk)
    
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(BOOKS_OUTPUT_PATH, table.schema, compression='snappy')
    writer.write_table(table)
    print(f'  Chunk {chunk_idx}: {len(chunk):,} rows (total: {total_rows:,})')

if writer:
    writer.close()

  Chunk 0: 50,000 rows (total: 50,000)
  Chunk 1: 50,000 rows (total: 100,000)
  Chunk 2: 50,000 rows (total: 150,000)
  Chunk 3: 50,000 rows (total: 200,000)
  Chunk 4: 50,000 rows (total: 250,000)
  Chunk 5: 50,000 rows (total: 300,000)
  Chunk 6: 2,935 rows (total: 302,935)


In [11]:
metadata = pq.read_metadata(BOOKS_OUTPUT_PATH)
print(f'Rows parquet: {metadata.num_rows:,}')
print(f'Columns: {metadata.num_columns}')
print(f'File size: {BOOKS_OUTPUT_PATH.stat().st_size / 1e6:.1f} MB')

df_check = pd.read_parquet(BOOKS_OUTPUT_PATH, columns=None).head(3)
print(f'\nFinal columns ({len(df_check.columns)}):')
print(list(df_check.columns))
display(df_check)

Rows parquet: 302,935
Columns: 14
File size: 197.9 MB

Final columns (12):
['text_reviews_count', 'series', 'language_code', 'popular_shelves', 'average_rating', 'authors', 'num_pages', 'publication_year', 'book_id', 'ratings_count', 'work_id', 'title']


,text_reviews_count,series,language_code,popular_shelves,average_rating,authors,num_pages,publication_year,book_id,ratings_count,work_id,title
0,7,[],,"[{'count': '56', 'name': 'to-read'}, {'count':...",4.13,"[{'author_id': '3041852', 'role': ''}]",162,2006,287141,46,278578,The Aeneid for Boys and Girls
1,15,[169353],,"[{'count': '159', 'name': 'to-read'}, {'count'...",3.93,"[{'author_id': '37778', 'role': ''}]",400,2009,6066814,186,6243149,"Crowner Royal (Crowner John Mystery, #13)"
2,615,[],,"[{'count': '4248', 'name': 'to-read'}, {'count...",3.98,"[{'author_id': '137561', 'role': ''}]",288,2006,89377,6949,86258,Penny from Heaven


# Interactions

In [12]:
INTER_SAMPLE_SIZE = 10_000
inter_sample = read_jsonl_sample(cfg.interactions_file, nrows=INTER_SAMPLE_SIZE)
print(f'Shape: {inter_sample.shape}')
print(f'Columns ({len(inter_sample.columns)}):')
print(list(inter_sample.columns))

Shape: (10000, 10)
Columns (10):
['user_id', 'book_id', 'review_id', 'is_read', 'rating', 'review_text_incomplete', 'date_added', 'date_updated', 'read_at', 'started_at']


In [13]:
summary_inter = pd.DataFrame({
    'dtype': inter_sample.dtypes.astype(str),
    'non_null': inter_sample.notna().sum(),
    'null_count': inter_sample.isna().sum(),
    'null_pct': (inter_sample.isna().sum() / len(inter_sample) * 100).round(1),
    'n_unique': inter_sample.apply(lambda col: col.map(str).nunique()),
    'example': inter_sample.iloc[0].astype(str).str[:80],
})
display(summary_inter)

,dtype,non_null,null_count,null_pct,n_unique,example
user_id,str,10000,0,0.0,191,8842281e1d1347389f2ab93d60773d4d
book_id,str,10000,0,0.0,6423,34684622
review_id,str,10000,0,0.0,10000,a53868823f065a0e20fd4ae98b820674
is_read,bool,10000,0,0.0,2,False
rating,int64,10000,0,0.0,6,0
review_text_incomplete,str,10000,0,0.0,805,
date_added,str,10000,0,0.0,9952,Tue Oct 17 09:40:11 -0700 2017
date_updated,str,10000,0,0.0,9930,Tue Oct 17 09:40:12 -0700 2017
read_at,str,10000,0,0.0,1673,
started_at,str,10000,0,0.0,1431,


In [14]:
inter_sample.head(3)

,user_id,book_id,review_id,is_read,rating,review_text_incomplete,date_added,date_updated,read_at,started_at
0,8842281e1d1347389f2ab93d60773d4d,34684622,a53868823f065a0e20fd4ae98b820674,False,0,,Tue Oct 17 09:40:11 -0700 2017,Tue Oct 17 09:40:12 -0700 2017,,
1,8842281e1d1347389f2ab93d60773d4d,34536488,9f08c5f991f87f3b7ae4ce779c2aac10,False,0,,Fri Oct 13 07:19:50 -0700 2017,Fri Oct 13 07:19:50 -0700 2017,,
2,8842281e1d1347389f2ab93d60773d4d,34467031,e66ba951e2147d34a275bd15c28aed11,False,0,,Tue Sep 26 21:22:01 -0700 2017,Tue Sep 26 21:22:01 -0700 2017,,


In [15]:
INTER_COLUMNS_TO_DROP = [
    'review_text_incomplete',
    'review_id',
    'read_at',
    'started_at',
]

In [16]:
inter_clean = inter_sample.drop(columns=[c for c in INTER_COLUMNS_TO_DROP if c in inter_sample.columns])

In [17]:
summary_inter_clean = pd.DataFrame({
    'dtype': inter_clean.dtypes.astype(str),
    'non_null': inter_clean.notna().sum(),
    'null_pct': (inter_clean.isna().sum() / len(inter_clean) * 100).round(1),
})
display(summary_inter_clean)

,dtype,non_null,null_pct
user_id,str,10000,0.0
book_id,str,10000,0.0
is_read,bool,10000,0.0
rating,int64,10000,0.0
date_added,str,10000,0.0
date_updated,str,10000,0.0


In [18]:
inter_clean.head(5)

,user_id,book_id,is_read,rating,date_added,date_updated
0,8842281e1d1347389f2ab93d60773d4d,34684622,False,0,Tue Oct 17 09:40:11 -0700 2017,Tue Oct 17 09:40:12 -0700 2017
1,8842281e1d1347389f2ab93d60773d4d,34536488,False,0,Fri Oct 13 07:19:50 -0700 2017,Fri Oct 13 07:19:50 -0700 2017
2,8842281e1d1347389f2ab93d60773d4d,34467031,False,0,Tue Sep 26 21:22:01 -0700 2017,Tue Sep 26 21:22:01 -0700 2017
3,8842281e1d1347389f2ab93d60773d4d,6383669,False,0,Sun Sep 24 21:14:51 -0700 2017,Sun Sep 24 21:14:51 -0700 2017
4,8842281e1d1347389f2ab93d60773d4d,34974754,False,0,Mon Aug 21 14:27:02 -0700 2017,Mon Aug 21 14:27:03 -0700 2017


In [19]:
from src.utils.io import read_jsonl_chunks
import pyarrow as pa
import pyarrow.parquet as pq

INTER_OUTPUT_PATH = OUTPUT_DIR / 'interactions_clean.parquet'
INTER_CHUNKSIZE = 500_000

writer = None
total_rows = 0

for chunk_idx, chunk in enumerate(read_jsonl_chunks(cfg.interactions_file, chunksize=INTER_CHUNKSIZE)):
    chunk = chunk.drop(columns=[c for c in INTER_COLUMNS_TO_DROP if c in chunk.columns])
    total_rows += len(chunk)
    
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(INTER_OUTPUT_PATH, table.schema, compression='snappy')
    writer.write_table(table)
    
    if chunk_idx % 10 == 0:
        print(f'  Chunk {chunk_idx}: {total_rows:,} rows so far')

if writer:
    writer.close()

print(f'   Total rows: {total_rows:,}')

  Chunk 0: 500,000 rows so far
  Chunk 10: 5,500,000 rows so far
  Chunk 20: 10,500,000 rows so far
  Chunk 30: 15,500,000 rows so far
  Chunk 40: 20,500,000 rows so far
  Chunk 50: 25,500,000 rows so far
  Chunk 60: 30,500,000 rows so far
   Total rows: 31,479,229


In [20]:
metadata = pq.read_metadata(INTER_OUTPUT_PATH)
print(f'Rows parquet: {metadata.num_rows:,}')
print(f'Columns: {metadata.num_columns}')
print(f'File size: {INTER_OUTPUT_PATH.stat().st_size / 1e6:.1f} MB')

df_check = pd.read_parquet(INTER_OUTPUT_PATH).head(3)
print(f'\nFinal columns ({len(df_check.columns)}):')
print(list(df_check.columns))
display(df_check)

Rows parquet: 31,479,229
Columns: 6
File size: 829.5 MB

Final columns (6):
['user_id', 'book_id', 'is_read', 'rating', 'date_added', 'date_updated']


,user_id,book_id,is_read,rating,date_added,date_updated
0,8842281e1d1347389f2ab93d60773d4d,34684622,False,0,Tue Oct 17 09:40:11 -0700 2017,Tue Oct 17 09:40:12 -0700 2017
1,8842281e1d1347389f2ab93d60773d4d,34536488,False,0,Fri Oct 13 07:19:50 -0700 2017,Fri Oct 13 07:19:50 -0700 2017
2,8842281e1d1347389f2ab93d60773d4d,34467031,False,0,Tue Sep 26 21:22:01 -0700 2017,Tue Sep 26 21:22:01 -0700 2017
